# 05 — Willingness to pay

**Purpose.** Estimate price sensitivity and direct-WTP for Aura at INR 199 per month.
**KPI validated.** Willingness to pay >= 60% at INR 199 per month (`plan.md` section 22).

Method: Van Westendorp Price Sensitivity Meter (4 questions Q23-Q26 in `quant_survey.md`) plus binary anchor at INR 199 (Q27). The four PSM curves cross at the Optimal Price Point (OPP), Indifference Price Point (IPP), Point of Marginal Cheapness (PMC), and Point of Marginal Expensiveness (PME).


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200
ACCENT = "#FF5B2E"

SEED = 20260507
rng = np.random.default_rng(SEED)
CHART_DIR = Path("../charts"); CHART_DIR.mkdir(exist_ok=True)
DERIVED = Path("../../derived")


## Data load

Real path: `pilot/derived/all_survey.csv` columns Q23 (too cheap), Q24 (bargain), Q25 (getting expensive), Q26 (too expensive), Q27 (yes/no/not sure at INR 199).


## SYNTHETIC DATA — REPLACE WITH REAL
Generates 30 participants with realistic Indian student INR thresholds: bargain centred around INR 99, getting-expensive around INR 249, too-expensive around INR 449, too-cheap around INR 39.


In [ ]:
def synth_psm(n=30):
    ids = [f"P{i:03d}" for i in range(1, n + 1)]
    rows = []
    for pid in ids:
        too_cheap = max(10, rng.normal(40, 12))
        bargain = max(too_cheap + 20, rng.normal(120, 35))
        expensive = max(bargain + 40, rng.normal(260, 70))
        too_expensive = max(expensive + 60, rng.normal(470, 140))
        # WTP at INR 199 — bias by where the participant lies
        yes_p = float(np.clip(0.55 + (180 - bargain) / 400, 0.15, 0.80))
        unsure_p = 0.15
        no_p = max(0.0, 1.0 - yes_p - unsure_p)
        probs = np.array([yes_p, no_p, unsure_p])
        probs = probs / probs.sum()
        wtp = rng.choice(["Yes", "No", "Not sure"], p=probs)
        rows.append({
            "participant_id": pid,
            "Q23_too_cheap": round(too_cheap),
            "Q24_bargain": round(bargain),
            "Q25_getting_expensive": round(expensive),
            "Q26_too_expensive": round(too_expensive),
            "Q27_pay_at_199": wtp,
        })
    return pd.DataFrame(rows)

if (DERIVED / "all_survey.csv").exists():
    survey = pd.read_csv(DERIVED / "all_survey.csv")
    psm = survey[["participant_id", "Q23_too_cheap", "Q24_bargain", "Q25_getting_expensive", "Q26_too_expensive", "Q27_pay_at_199"]].dropna()
else:
    psm = synth_psm()
print("rows:", len(psm))
psm.head()


## Binary WTP at INR 199


In [ ]:
wtp_counts = psm["Q27_pay_at_199"].value_counts().reindex(["Yes", "No", "Not sure"]).fillna(0).astype(int)
n = int(wtp_counts.sum())
yes_pct = wtp_counts.get("Yes", 0) / n * 100
# 95% Wilson-style CI on the binomial (use normal approx for small n)
p = yes_pct / 100
se = np.sqrt(p * (1 - p) / n)
lo, hi = max(0, (p - 1.96 * se) * 100), min(100, (p + 1.96 * se) * 100)
print(wtp_counts.to_string())
print(f"\nYes at INR 199 = {yes_pct:.1f}%  (95% CI [{lo:.1f}%, {hi:.1f}%], n={n})")
print(f"meets >=60% target: {lo >= 60}")


## Van Westendorp curves

Build cumulative distributions for each of the four price questions across the price grid. Standard PSM convention:
- "Too cheap" decreasing — share who say price is too cheap (quality concern) at or above P.
- "Bargain" decreasing — share who say price is a bargain at or below P (so we use 1 - cumulative).
- "Getting expensive" increasing — share who say price is starting to feel expensive at or below P.
- "Too expensive" increasing — share who say price is too expensive at or below P.

Crossings:
- **OPP** = "too cheap" cross "too expensive" (Optimal Price Point).
- **IPP** = "bargain" cross "getting expensive" (Indifference Price Point).
- **PMC** = "too cheap" cross "getting expensive" (Point of Marginal Cheapness, lower bound).
- **PME** = "bargain" cross "too expensive" (Point of Marginal Expensiveness, upper bound).


In [ ]:
prices = np.arange(0, 1001, 5)

def cum_at_or_below(values, grid):
    v = np.asarray(values, dtype=float)
    return np.array([(v <= p).mean() for p in grid])

def cum_at_or_above(values, grid):
    v = np.asarray(values, dtype=float)
    return np.array([(v >= p).mean() for p in grid])

too_cheap = cum_at_or_above(psm["Q23_too_cheap"], prices)        # decreasing
bargain = cum_at_or_above(psm["Q24_bargain"], prices)             # decreasing
getting_exp = cum_at_or_below(psm["Q25_getting_expensive"], prices)  # increasing
too_exp = cum_at_or_below(psm["Q26_too_expensive"], prices)          # increasing

def crossing(a, b, grid):
    diff = a - b
    sign_change = np.where(np.diff(np.sign(diff)) != 0)[0]
    if len(sign_change) == 0:
        return None
    i = sign_change[0]
    return float(grid[i])

opp = crossing(too_cheap, too_exp, prices)
ipp = crossing(bargain, getting_exp, prices)
pmc = crossing(too_cheap, getting_exp, prices)
pme = crossing(bargain, too_exp, prices)
print(f"OPP (Optimal):                INR {opp}")
print(f"IPP (Indifference):           INR {ipp}")
print(f"PMC (Marginal Cheapness):     INR {pmc}")
print(f"PME (Marginal Expensiveness): INR {pme}")
print(f"Acceptable price range: [{pmc}, {pme}]")


## Results — interpretation

- The acceptable price range runs from PMC (any cheaper sounds suspicious) to PME (any pricier kills demand).
- OPP is the single-point estimate where "too cheap" equals "too expensive" — usually the launch price recommendation.
- The binary WTP at INR 199 is the direct read against the brief's KPI. A simple yes-share with a 95% CI satisfies the reporting standard.
- Synthetic OPP tends to land in the INR 130-180 band, suggesting INR 199 is a slight stretch on this cohort — useful framing for the pricing slide.


## Charts

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.8))
ax.plot(prices, too_cheap * 100, label="Too cheap", color="#0E0E0E", linestyle="--")
ax.plot(prices, bargain * 100, label="Bargain", color="#0E0E0E", linestyle="-")
ax.plot(prices, getting_exp * 100, label="Getting expensive", color=ACCENT, linestyle="-")
ax.plot(prices, too_exp * 100, label="Too expensive", color=ACCENT, linestyle="--")
for x, lab in [(opp, "OPP"), (ipp, "IPP"), (pmc, "PMC"), (pme, "PME")]:
    if x is not None:
        ax.axvline(x, color="grey", linewidth=0.6)
        ax.text(x, 96, lab, ha="center", fontsize=9)
ax.axvline(199, color=ACCENT, linewidth=1.4, alpha=0.6)
ax.text(199, 6, "INR 199", color=ACCENT, ha="center", fontsize=9)
ax.set_xlabel("Price (INR per month)")
ax.set_ylabel("Share of participants (%)")
ax.set_title("Van Westendorp price sensitivity meter")
ax.set_xlim(0, 1000)
ax.set_ylim(0, 100)
ax.legend(loc="center right")
fig.tight_layout()
fig.savefig(CHART_DIR / "05_van_westendorp.png")
plt.show()

fig, ax = plt.subplots(figsize=(5.6, 4.0))
ax.bar(wtp_counts.index, wtp_counts.values, color=[ACCENT, "#0E0E0E", "grey"], alpha=0.9)
for i, v in enumerate(wtp_counts.values):
    ax.text(i, v + 0.3, str(int(v)), ha="center")
ax.set_ylabel("Participants")
ax.set_title(f"Would pay INR 199/month  ({yes_pct:.1f}% yes, n={n})")
fig.tight_layout()
fig.savefig(CHART_DIR / "05_wtp_binary.png")
plt.show()
